# Isoprene Absorption Spectrum

---

Last updated: March 6, 2025 (JY)

This notebook calculates Voigt line broadening on the isoprene absorption spectrum and compares it to experimental spectra from HITRAN. This notebook is coded in Julia.

In [49]:
# Activate Julia packages
using Pkg;
Pkg.activate("../"); # Activates the vSmartMOM.jl Project.toml file
Pkg.instantiate();

# Using local module, since the current version of vSmartMOM does not support isoprene (not a HITRAN line list species)
include("../src/vSmartMOM.jl");
include("helper_functions.jl");

# Other imports
using .vSmartMOM, .vSmartMOM.Absorption
using Revise, PlotlyBase, Plots, LinearAlgebra, NCDatasets, Format, LaTeXStrings;
using FiniteDiff;

default(fontfamily="Computer Modern");

  Activating project at `~/Documents/Radiative_Transfer/vSmartMOM.jl`


In [6]:
filepath = "../src/Absorption/constants/pseudo_line_lists/isoprene/c5h8_isoprene.101"
ν_min = 890.
ν_max = 910.

isoprene_hitran = vSmartMOM.Absorption.read_hitran_isoprene(filepath, ν_min = ν_min, ν_max = ν_max);

lineModel = vSmartMOM.Absorption.Voigt()
isoprene_voigt = vSmartMOM.make_hitran_model(isoprene_hitran, lineModel, wing_cutoff=10, architecture=vSmartMOM.CPU());
hitran_array = [isoprene_voigt];

In [7]:
date = "20190701"
startTime = "06z" # 00, 06, 12 or 18 in UTC, i.e. 6 hourly data stacked together

lat = 2.
lon = 100.

# Here, we download a reanalysis file from MERRA (NASA), these files are huge (500Mb), so it might take a file. We are using JULIA tools again, usually you will have to download these yourself:
MerraFile = "MERRA2_400.inst6_3d_ana_Nv." * date * ".nc4"
MerraFolder = "MERRA2/"
MF = joinpath(MerraFolder, MerraFile)
profile_hr = read_atmos_profile(MF, lat, lon, 4);

pressures = profile_hr.p;
temperatures = profile_hr.T;

ds["T"] = T (576 × 361 × 72 × 4)
  Datatype:    Union{Missing, Float32} (Float32)
  Dimensions:  lon × lat × lev × time
  Attributes:
   long_name            = Air temperature
   units                = K
   _FillValue           = 1.0e15
   missing_value        = 1.0e15
   fmissing_value       = 1.0e15
   scale_factor         = 1.0
   add_offset           = 0.0
   standard_name        = air_temperature
   vmax                 = 1.0e15
   vmin                 = -1.0e15
   valid_range          = Float32[-1.0f15, 1.0f15]



In [8]:
res = 0.01;
ν_grid = ν_min:res:ν_max;

model_interp = compute_profile_crossSections_isoprene(profile_hr, hitran_array, ν_grid);

In [60]:
scaling = 10^18

index = 45
p = plot(ν_grid, model_interp[:,index,1] * scaling,
    title = L"$\mathrm{C_5H_8}$ Absorption Spectrum: Pseudo Line List", label = "P = $(round(pressures[index] / 100, digits = 2)) hPa, T = $(round(temperatures[index], digits = 1)) K", lw = 2, dpi = 600)

index = 50
plot!(ν_grid, model_interp[:,index,1] * scaling,
    label = "P = $(round(pressures[index] / 100, digits = 2)) hPa, T = $(round(temperatures[index], digits = 2)) K", lw = 2)

index = 72
plot!(ν_grid, model_interp[:,index,1] * scaling,
    label = "P = $(round(pressures[index] / 100, digits = 2)) hPa, T = $(round(temperatures[index], digits = 2)) K", lw = 2)

ylabel!(L"Absorption Cross Section $\mathrm{[x 10^{-18} cm^{-2} molec^{-1}]}$")
xlabel!(L"Wavenumber $\mathrm{[cm^{-1}]}$")
savefig(p, "figures/absorption_cross_section.png")

"/Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM.jl/isoprene_scripts/figures/absorption_cross_section.png"

## Compare Voigt-broadened absorption spectra with PNNL spectra

In [38]:
resolution = (6499.9532 - 599.9943) / 97909 # 97902
nu = collect(599.9943:resolution:6499.9532)

experimental_278 = vec(readdlm("../src/Absorption/constants/pseudo_line_lists/isoprene/C5-H8_278.1K-760.0K_600.0-6500.0_0.11_N2_505_43.xsc.txt", ' ')[:,2:end]');
experimental_298 = vec(readdlm("../src/Absorption/constants/pseudo_line_lists/isoprene/C5-H8_298.1K-760.0K_600.0-6500.0_0.11_N2_505_43.xsc.txt", ' ')[:,2:end]');
experimental_323 = vec(readdlm("../src/Absorption/constants/pseudo_line_lists/isoprene/C5-H8_323.1K-760.0K_600.0-6500.0_0.11_N2_505_43.xsc.txt", ' ')[:,2:end]');

# Modeled Spectra
modeled_spectra_278 = vSmartMOM.Absorption.absorption_cross_section_isoprene(isoprene_voigt, 
    ν_grid, 
    1013.25, 
    278.1)
compute_profile_crossSections_isoprene(profile_hr, hitran_array, ν_grid);

modeled_spectra_298 = vSmartMOM.Absorption.absorption_cross_section_isoprene(isoprene_voigt, 
    ν_grid, 
    1013.25, 
    298.1)
compute_profile_crossSections_isoprene(profile_hr, hitran_array, ν_grid);

modeled_spectra_323 = vSmartMOM.Absorption.absorption_cross_section_isoprene(isoprene_voigt, 
    ν_grid, 
    1013.25, 
    323.1)
compute_profile_crossSections_isoprene(profile_hr, hitran_array, ν_grid);

In [56]:
scaling = 10^18

p1 = plot(nu, experimental_278 * scaling, xlimit = [890, 910], label = "Experimental (278 K)", lw = 2, legend = :outertopright, dpi = 600, color = "black")
plot!(ν_grid, modeled_spectra_278 * scaling, xlimit = [890, 910], label = "Simulated (278 K)", lw = 2)
title!(L"$\mathrm{C_5H_8}$ Experimental/Simulated Spectra")

p2 = plot(nu, experimental_298 * scaling, xlimit = [890, 910], label = "Experimental (298 K)", lw = 2, legend = :outertopright, color = "black")
plot!(ν_grid, modeled_spectra_298 * scaling, xlimit = [890, 910], label = "Simulated (298 K)", lw = 2)
ylabel!(L"Absorption Cross Section $\mathrm{[x 10^{-18} cm^{-2} molec^{-1}]}$")

p3 = plot(nu, experimental_323 * scaling, xlimit = [890, 910], label = "Experimental (323 K)", lw = 2, legend = :outertopright, color = "black")
plot!(ν_grid, modeled_spectra_323 * scaling, xlimit = [890, 910], label = "Simulated (323 K)", lw = 2)
xlabel!(L"Wavenumber $\mathrm{[cm^{-1}]}$")

p = plot(p1, p2, p3, layout = (3,1))
savefig(p, "figures/absorption_cross_section_vs_experimental.png")

"/Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM.jl/isoprene_scripts/figures/absorption_cross_section_vs_experimental.png"